In [1]:
import os
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
import statistics as stat
import math
import sympy as sym
from sympy import solve
from scipy.optimize import dual_annealing

from scipy.optimize import minimize
from scipy.optimize import curve_fit
from scipy.optimize import least_squares 

from operator import itemgetter

### User-Defined Inputs

In [2]:
# Material and root directory
mat = 'Phrozen'
dir = '/Users/mikelapera/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/JHU/Research/Experiments/DMA/data/Phrozen'
fid3 = 'Phrozen_DC_35C-160C_cleaned.csv'
filepath = os.path.join(dir,fid3)

# Parse tests run at different temperatures using angular frequency patterns
inc = 7 # number of data points taken per temperature

# Universal Gas Constant
R = 8314.5

# Reference temperature (deg. C)
T0 = '90C'

# DMA temperature increments - typically 5 or 10 depending
deg_inc = 5

# DMA angular angular frequency increments
freq_inc = 7

# Number of Prony terms for Prony series - approx. 1 term per decade of master curve
num_prony_terms = 30

# Poisson's ratio (room temperature)
nu = 0.34

# Number of knots for spline fit of master curve (helps prevent spline overfitting/instability)
# when using scipy UnivariateSpline
num_knots = 1

# Number of terms by which to evaluate spline fit for master curve (new x range)
num_spl_terms = 20

# Number of terms for polynomial fit for master curve
num_terms = 15

# Fitting function for mastercuve (spline, exp) - 'exp' option no longer supported
mastercurve_fit_type = 'spline'

# Shift type - manual vs. auto
shift_type = 'manual'

# If need to remove data from master curve to improve fitting, set to True
MC_red = True

# Time-Temperature Shifting Relationship
TRS = 'WLF'

### Functions

In [3]:
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

def mouse_event(event):
    print('x: {} and y: {}'.format(event.xdata, event.ydata))
    xval = event.xdata
    yval = event.ydata
    # Use find_nearest to locate index of storage modulus value closest to
    # click value
    ind = np.where(df3['Storage modulus (Mpa)'].to_numpy()==find_nearest(
        df3['Storage modulus (Mpa)'].to_numpy(),(yval)))[0][0]
    # Use index to identify temperature associated with storage modulus
    temp_val = df3['Temperature (degC)'].iloc[ind]
    freq_val = df3['Angular frequency (rad/s)'].iloc[ind]
    coords = [xval,]
def round2place(x,base):
    return base * int(round(x/base))


########################### Partition Isotherms ################################################

def split_isotherms_func(T0,Ep_isotherm,Ep_isotherm_low,Ep_isotherm_high):
    
    templist = list(dict.keys(Ep_isotherm))
    Tref_index = templist.index(T0)
    T_low = templist[:templist.index(T0)]
    T_low.reverse()
    T_low.insert(0,T0)
    T_high = Tlow = templist[templist.index(T0):]
    # Partion isotherms based on whether they are above or below Tref
    for i in T_low:
        Ep_isotherm_low[i] = Ep_isotherm[i]
    for i in T_high:
        Ep_isotherm_high[i] = Ep_isotherm[i]

    
    return T_low, T_high, Ep_isotherm_low, Ep_isotherm_high        

####################################### WLF Fit #################################################

def WLF_func(x,C1,C2):
    y = -1 * ((C1 * x)/(C2 + x))
    return y

def WLF_curvefit(Tdiff,alpha,T0):
    Tdiff = np.asarray(Tdiff)
    log_alpha = np.asarray(alpha)
    parameters, covariance = curve_fit(WLF_func, Tdiff, log_alpha)
    fit_C1 = parameters[0]
    fit_C2 = parameters[1]
    
    fit_y = WLF_func(Tdiff,fit_C1,fit_C2)
    
    # Plotting

    plt.figure(5)
    plt.plot(Tdiff,log_alpha,'o', label = 'calculated shift factors')
    plt.plot(Tdiff, fit_y, '-', label = 'WLF Fit')
    plt.legend()
    plt.title('WLF Plot for $T_{0}$ = ' + T0)   
    plt.xlabel('$T - T_{0}$')   
    plt.ylabel('$log(a_{T})$')
    plt.grid(which='both',axis='both')

    return fit_C1, fit_C2, fit_y, log_alpha    

####################################### Arrhenius Fit ###########################################
def Arrhenius_func(x,Q):
    #T0 = reference temperature
    y = (Q / R) * (x - (1 / (Tref + 273)))
    return y

def Arrhenius_curvefit(temps_K,alpha,T0):
    temps_K = np.asarray(temps_K)
    global Tref, R
    Tref = int(T0[:-1])
    T = (1 / temps_K)
    # Natural log of alpha
    ln_alpha = np.log(10**np.asarray(alpha))
    parameters, covariance = curve_fit(Arrhenius_func, T, ln_alpha)
    fit_Q = parameters[0]
    
    fitted_y = Arrhenius_func(T,fit_Q)

    
    plt.figure(5)
    plt.plot(T,ln_alpha,'o', label = 'calculated shift factors')
    plt.plot(T, fitted_y, '-', label = 'Arrhenius Fit')
    plt.legend()
    plt.title('Arrhenius Plot for $T_{0}$ = ' + T0)   
    plt.xlabel('1/T (1/K)')   
    plt.ylabel('$ln(a_{T})$')
    plt.grid(which='both',axis='both')
    return fit_Q

########################### Fit Prony Series IAW Abaqus Definition ###############################
'''
Notes:
1. e_prony -- a vector of modulus ratios for Prony series. Length equal to p
2. p       -- a vector of time constants (tau) for Prony series. This can be manually selected a priori
              and typically has a length equal to the number of decades of the master curve
3. xdata   -- frequency (x-axis) values of master curve
4. ydata   -- storage modulus (y-axis) values of master curve
5. scipy.optimize.least_squares is used to minimize fun_relax via fun_res
'''
def fit_prony(E0,x3,y3,num_prony_terms):

    def fun_relax(e_prony,x3):
        x = x3
        Eprime1 = (1 - sum((e_prony[i]) for i in range(len(p))))
        Eprime2 = sum((e_prony[i] * p[i]**2 * np.power(x,2))/(1 + p[i]**2 * np.power(x,2)) for i in range(len(p)))
        return Eprime1 + Eprime2
        
    def fun_res(e_prony):
        return fun_relax(e_prony,x3) - Ep_data
            
    # Set relaxation spectrum storage modulus values as y-values
    Ep_data = y3/E0        
    p = np.array([])
    e_prony = np.array([])

    for j in (range(0,num_prony_terms)):
        min_time = np.floor(-num_prony_terms/2)
        #min_time = -4
        #E0.append(10**(min_time + int(j)))
        # Initial guess in which all values will be between 0 and 1 IAW Abaqus implementation
        e_prony = np.append(e_prony, 0.5)
        p = np.append(p,10**(min_time + int(j)))
    #p = np.array([1E7,1E8,1E9,1E10,1E11])
    #e_prony = np.ones(p.shape[0],) * 0.5
    e_prony = np.ones(p.shape[0],) * 0.5


    # Least squares 
    ls_res = least_squares(fun_res, e_prony, bounds = (0,1))
    output = fun_relax(ls_res.x,x3)
    # Return storage modulus approximations made by Prony series
    ei = E0 * fun_relax(ls_res.x,x3)

    plt.figure()
    plt.loglog(x3,y3,'o')
    plt.loglog(x3,E0 * fun_relax(ls_res.x,x3),'-')
    plt.grid(which = 'both')
    plt.legend(['Polynomial Fit to Experimental DMA Master Curve','Prony Series Approximation of DMA Master Curve'])
    plt.xlabel('$\omega$')
    plt.ylabel('Storage Modulus (MPa)')
    return ls_res.x  

########################### Fit Polynomial Function to Master Curve ###############################

def fit_poly_func(log_omega,log_Ep,num_terms):
    # Flatten list of lists to single list
    x1 = []
    y1 = []
    for i in log_omega:
        x1.extend(i)
    for j in log_Ep:
        y1.extend(j)
    # Make list of lists using coordinate pairs i.e. [[x1,y1],[x2,y2],[x3,y3],...]
    xy=[]
    for i in range(0,len(x1)):
            xy.append([x1[i],y1[i]])
    # Sort coordinate pairs in ascending order WRT x values
    xy_sorted = sorted(xy,key=itemgetter(0))
    x2 = []
    y2 = []
    # Build single list for x and y (individually) in ascending order 
    # WRT x while retaining/honoring coordinate pairs
    for i in xy_sorted:
        x2.append(i[0])
        y2.append(i[1])
    # Fit spline to master curve
    #xnew =  np.linspace(np.floor(np.min(np.asarray(x2))),np.ceil(np.max(np.asarray(x2))),num_terms)
    xnew =  np.linspace(np.asarray(x2)[0],np.asarray(x2)[-1],num_terms*10)
    # Calculate polynomial coeffs
    pp = np.polyfit(x2,y2,num_terms)
    # Build fitted polynomial using xnew
    predict = np.poly1d(pp)
    # Evaluate fitted polynomial with xnew
    ynew = predict(xnew)
    # Plot original test data and overlay spline fit
    plt.figure()
    plt.plot(x2, y2, 'o')
    plt.plot(xnew, ynew, 'r-')
    plt.xlabel(r'log($\omega$ * $a_{T})$')
    plt.ylabel("log(E')")
    plt.legend(['original data','Polynomial fit, n = ' + str(num_terms)])
    plt.grid()
    poly_output = np.poly1d(pp)(xnew)
    
    return x2, y2, xnew, ynew, poly_output 

########################### Generate time domain relaxation modulus from Prony Series ###############################

def relaxation_modulus_abq(p,E0,e_prony,t):
    E_r = E0 * (1 - sum((Ei[i] * (1 - np.exp(-t/p[i]))) for i in range(len(p))))
    return E_r

########################### Manual shifting functions for shifting isotherms ###############################

def plot_isotherms(x_all,y_all,temp1,temp0):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.set_title('Select a point from each dataset @ approx. same y-value' + ' ----- Shifting ' + temp1 + ' to ' + temp0)
    points = []
    line, = ax.plot(x_all,y_all,'o', picker=n)
    fig.canvas.mpl_connect('pick_event', onpick)
    plt.grid()
    plt.pause(10)
    plt.show()

def onpick(event):
    if len(points) < n:
        thisline = event.artist
        xdata = thisline.get_xdata()
        ydata = thisline.get_ydata()
        ind = event.ind
        point = tuple(zip(xdata[ind], ydata[ind]))
        points.append(point)
        print('onpick point:', point)
    else:
        print('already have {} points'.format(len(points)))


def manual_shift(Ep_isotherm,Tref,temp_range,legstr3,Ep_isotherm_shifted,shift_factors,all_Ep,all_omega):
    # Initialize dictionaries with reference shift factor, storage modulus isotherm, and legend entry on first function call
    data={}
    if shift_factors:
        pass
    else:
        shift_factors = {}
        shift_factors[Tref] = 0
    if Ep_isotherm:
        pass
    else:
        Ep_isotherm_shifted = {}
        Ep_isotherm_shifted[Tref] = [Ep_isotherm[Tref][0], Ep_isotherm[Tref][1]]  
    if legstr3:     
        pass
    else:
        legstr3= []
        legstr3.append('Tref = ' + Tref)
    for i in range(1,len(temp_range)):
        print("i = " + str(i))
        # Establish global declaration for isotherm data for passing into obj_func_hz2(v)
        global x1, y1, x2, y2, points
    
        # Temperature ratio to be multiplied by storage modulus for plotting mastercurve (See Ferry derivation)
        # Convert to absolute temperature
        #temp_ratio = float("{:.2f}".format((int((T0[:-1]))+273)/(int((temp_range[i])[:-1])+273)))
        temp_ratio = 1
        
        print('Shifting ' + temp_range[i] + '...')
        # Ep_isotherm_shifted is incrementally built using Ep_isotherm_low and Ep_isotherm_high
        x1 = np.log10(Ep_isotherm_shifted[temp_range[i-1]][0])
        y1 = np.log10(Ep_isotherm_shifted[temp_range[i-1]][1] * temp_ratio)   
        # Unshifted "partitioned" Ep_isotherm (Ep_isotherm_high or Ep_isotherm_low)
        x2 = np.log10(Ep_isotherm[temp_range[i]][0])
        y2 = np.log10(Ep_isotherm[temp_range[i]][1] * temp_ratio)
        x_all = np.append(x1,x2)
        y_all = np.append(y1,y2)
        
        points = []
        label = temp_range[i]
        plot_isotherms(x_all,y_all,temp_range[i],temp_range[i-1])
        data[label] = points
        Tref_int = int((Tref[:-1]))
        T1_int = int((temp_range[i])[:-1])
        log_aT, shift_factors = shift_factor_calc(data,temp_range[i],T1_int,Tref_int,shift_factors)
        # Store storage modulus and frequency values
        Ep_isotherm_shifted[temp_range[i]] = [Ep_isotherm[temp_range[i]][0] * 10**log_aT, Ep_isotherm[temp_range[i]][1] * temp_ratio]
        # Append legend
        legstr3.append(temp_range[i])
    
    # for k in temp_range:
    #     plt.figure(len(temp_range)+1)
    #     plt.plot(np.log10(Ep_isotherm_shifted[k][0]),np.log10(Ep_isotherm_shifted[k][1]))
    # plt.legend(legstr3)
    # plt.xlabel(r'log($\omega * a_{T}$)')
    # plt.ylabel("Log(E') (MPa)")
    # plt.grid(which='major',axis='both')
    # plt.show()
    # Add reference temperature isotherm to dictionaries and plot

    
    return shift_factors, Ep_isotherm_shifted, legstr3
    
def shift_factor_calc(data,temp,T1_int,Tref_int,shift_factors):    
        a = list(zip(*data[temp]))
        dx = a[0][0][0]-a[0][1][0]
        dy = a[0][0][1]-a[0][1][1]
        if T1_int < Tref_int:
            log_aT = abs(dx)
        else:
            log_aT = abs(dx) * -1
        shift_factors[temp] = log_aT
        print(shift_factors[temp])
        return log_aT, shift_factors

### --- Main --- 

#### Initializations

In [4]:
file3 = pd.read_csv(filepath)

################ INITIALIZE NECESSARY LISTS AND DICTIONARIES #########################

# Shift factors
shift_factors = {}
# Legend lists
legstr = []
legstr2 = []
legstr3 = []

# Lists need for WLF fit
Tdiff = []
alpha = []

# All shifted data that comprises master curve for later fitting of Prony series
all_Ep = []
all_omega = []
Ep_isotherm_shifted = {}
Ep_isotherm_low = {}
Ep_isotherm_high = {}

E2 = []
temp2 = []
coords = []
Ep_isotherm = {}
Epp_isotherm = {}
Ep_isofreq = {}
Epp_isofreq = {}
temp_sample = []
freq_sampleRPS = []
freq_sampleHz = []

%matplotlib qt

Read in DMA Data and Build Dictionary

In [ ]:
for i in range(0,file3.shape[0],inc):
    test_name = 'Test_' + str(file3['Temperature (degC)'][i])
    df3 = {}
    df3 = file3[i:i+inc][:]
    df3.to_csv(test_name + '.csv')
    temp = stat.mean(df3['Temperature (degC)'].to_numpy())
    temp = round2place(temp,deg_inc)
    temp_key = str(temp) + 'C'
    legstr.append(str(temp) + ' C')
    
    
    Eprime = df3['Storage modulus (Mpa)'].to_numpy()
    #TanD = df3['Tan(delta)'].to_numpy()
    omega = df3['Angular frequency (rad/s)'].to_numpy()
    freq1 = df3['Angular frequency (rad/s)'].to_numpy() / (2*math.pi)
    Ep_isotherm[temp_key] = [omega,Eprime]
    
    plt.figure(1)
    plt.plot(np.log10(freq1),np.log10(Eprime),'-o')
    plt.show()
    #plt.semilogx(omega,TanD,'-o')

plt.title("Storage Modulus vs. Frequency " + '(' + mat +')')
#plt.title("Tan Delta vs. Frequency")
plt.xlabel("log($\omega$)")
plt.ylabel("log(E') (MPa)")
plt.grid(which = 'both', axis = 'both') 
#plt.ylabel("Tan Delta")
plt.legend(legstr,loc='center left', bbox_to_anchor=(1.04,0.5), fontsize = 10)
plt.savefig('Eprime_v_Freq.png')
#plt.savefig('TanD_v_Freq.png')


# Build array of frequencies from which to sample from based on experimental setup 
freq_sampleRPS = Ep_isotherm[T0][0][:].round()
freq_sampleHz = (Ep_isotherm[T0][0][:] / (2 * math.pi)).round()

# Build for storage modulus vs. temperature isofrequency dictionaries
for i in range(0,freq_sampleRPS.shape[0]):  
    T_temp_list = []
    Ep_temp_list = []
    # Key
    freqRPS = str(int(freq_sampleRPS[i])) + ' rps'
    freqHz  = str(int(freq_sampleHz[i])) + ' Hz'
    ctr = 0
    for j in list(Ep_isotherm.keys()):
        
        # Temperature of Ep_isotherm
        T = int(list(Ep_isotherm.keys())[ctr][:-1])
        
        # Locate index of frequency within Ep_isotherm dictionary
        ind = np.where(Ep_isotherm[j][0]==find_nearest(Ep_isotherm[j][0],freq_sampleRPS[i]))[0][0]
        
        # Frequency value and storage modulus value store
        T_temp_list.append(T)
        Ep_temp_list.append(Ep_isotherm[j][1][ind])
        ctr += 1
        
    Ep_isofreq[freqRPS] = [T_temp_list,Ep_temp_list] 
    legstr2.append(str(freqHz))
    plt.figure(2)
    plt.plot(Ep_isofreq[freqRPS][0],list(np.asarray(Ep_isofreq[freqRPS][1])),'-o')
    
plt.figure(2)
plt.show
plt.legend(legstr2,loc='center left', bbox_to_anchor=(1.04,0.5))
plt.xlabel("Temperature (deg C)")
plt.ylabel("Storage Modulus (MPa)")
plt.title("Storage Modulus vs. Temperature " + '(' + mat +')')
plt.grid()
plt.savefig('Ep_v_Temp_isofreq.png')

: 

#### Construct Master Curve using Time-Temperature-Superposition (TTS)

In [ ]:
plt.close("all")
# Populate shifted storage modulus dictionary with reference curve since it is
# not shifted
n=2
data={}
# Ep_isotherm_shifted is incrementally built using Ep_isotherm_low and Ep_isotherm_high
# in arc_length_minimization_v2
xref = Ep_isotherm[T0][0]
yref = Ep_isotherm[T0][1]
Ep_isotherm_shifted[T0] = [xref,yref]

# Split up isotherm data with respect to reference temperature
T_low, T_high, Ep_isotherm_low, Ep_isotherm_high = split_isotherms_func(T0,Ep_isotherm,Ep_isotherm_low,Ep_isotherm_high)

# Manually shift isotherms
T_eval = [T_low, T_high]
for z in T_eval: 
    shift_factors, Ep_isotherm_shifted, legstr3 =  manual_shift(Ep_isotherm,T0,z,legstr3,Ep_isotherm_shifted,shift_factors,all_Ep,all_omega)

plt.close('all')

# Plot all shifted data
temp_list = list(Ep_isotherm.keys())
legstr4 = []
for k in temp_list:
    plt.figure(1)
    plt.plot(np.log10(Ep_isotherm_shifted[k][0]),np.log10(Ep_isotherm_shifted[k][1]))
    if k == T0:
        legstr4.append(k + ' ($T_{ref}$)')
    else: 
        legstr4.append(k)
plt.legend(legstr4)
plt.xlabel(r'log($\omega * a_{T}$)')
plt.ylabel("Log(E') (MPa)")
plt.grid(which='major',axis='both')
plt.show()
		
plt.close(plt.figure(2))
for k in temp_list:
    plt.figure(2)
    plt.plot(int(k[:-1])-int(T0[:-1]),shift_factors[k],'o')
    legstr4.append(k)
plt.legend(legstr4)
plt.xlabel('(T - T0)')
plt.ylabel('$log(a_{T}$)')
plt.title('Horizontal Shift Factors')
plt.grid(which='major',axis='both')
plt.show()	
    
# Pickle results and save to directory containing DMA data
Ep_name = os.path.join(dir,'storage_modulus_' + mat + '.pickle')
master_name = os.path.join(dir,'master_curve_' + mat + '.pickle')
SF_name = os.path.join(dir,'shift_factors_' + mat + '.pickle')
with open(Ep_name, 'wb') as handle:
    pickle.dump(Ep_isotherm, handle, protocol = pickle.HIGHEST_PROTOCOL)
with open(master_name, 'wb') as handle:
    pickle.dump(Ep_isotherm_shifted, handle, protocol = pickle.HIGHEST_PROTOCOL)   
with open(SF_name, 'wb') as handle:
    pickle.dump(shift_factors, handle, protocol = pickle.HIGHEST_PROTOCOL)  


#### Calculate WLF or Arrhenius Fit Using Calculated Shift Factors

$$
\text{log}(\alpha_T) = \frac{-C_1(T - T_0)}{C_2 + (T-T_0)}
$$

#### If need to trim shift factor data, activate block (this often improves WLF fit) and deactivate subsequent block
This will keep shift factors greater than reference temperature (T0)

In [ ]:
all_ind = list(shift_factors.keys())
new_ind = []
for i in all_ind:
    new_ind.append(int(i[:-1]))

IDs = np.asarray(new_ind).reshape(len(new_ind),1) - int(T0[:-1])*np.ones([len(new_ind),1]) >= 0
pos_ind = [i for i, val in enumerate(IDs) if val]
pos_ind_keys = []

for i in pos_ind:
    pos_ind_keys.append(str(all_ind[i]))

if TRS == 'WLF':
    Tdiff_red = []
    for i in pos_ind_keys:
        Tdiff_red.append(int(i[:-1]) - int(T0[:-1]))

    alpha_red = [shift_factors[x] for x in pos_ind_keys]

    fit_C1, fit_C2, fit_y, log_alpha = WLF_curvefit(Tdiff_red,alpha_red,T0)
    print(format(fit_C1,".2f"))
    print(format(fit_C2,".2f"))
    
elif TRS == 'Arrhenius':
    T_red = []
    # Conver temperatures to Kelvin
    for i in pos_ind_keys:
        T_red.append(int(i[:-1]) + 273)

    alpha_red = [shift_factors[x] for x in pos_ind_keys]

    fit_Q = Arrhenius_curvefit(T_red, alpha_red,T0)
    print(format(fit_Q,".2e"))

In [ ]:
# # Interpret results dictionary output to build plot to fit WLF equation
# Tdiff = []
# alpha = []
# if TRS == 'WLF':
#     for i in shift_factors:
#         Tdiff.append(int(i[:-1]) - int(T0[:-1]))
#         alpha.append(shift_factors[i])
#     # Calculate WLF constants and plot results
#     fit_C1, fit_C2, fit_y, log_alpha = WLF_curvefit(Tdiff,alpha,T0)
#     print(fit_C1)
#     print(fit_C2)
    
# elif TRS == 'Arrhenius':
#     # Calculate Arrhenius activation energy (in lieu of WLF)
#     # Array of test temperatures (absolute)
#     alpha = []
#     temps_K = []
#     for i in shift_factors:
#         temps_K.append(int(i[:-1]) + 273)
#         alpha.append(shift_factors[i])
#     fit_Q = Arrhenius_curvefit(temps_K, alpha,T0)

# else:
#     print('Shift function not supported')

#### If need to remove data from master curve to improve fitting, specify which isotherms to remove


In [378]:
if MC_red == True:
    Ep_data_all = Ep_isotherm_shifted
    del Ep_data_all['170C']
    del Ep_data_all['180C']
    del Ep_data_all['190C']
    del Ep_data_all['200C']

    for k in list(Ep_data_all.keys()):
        plt.plot(np.log10(Ep_data_all[k][0],np.log10(Ep_data_all[k][1])))
    plt.show()
    plt.xlabel('log($\omega * \alpha_T$)')
    plt.ylabel('log(Storage Modulus) [MPa]')


True

### Fit Polynomial to Master Curve

In [379]:
# Fit curve to master curve
# x1, y1     : "listized" Eprime vs. log(omega)*alpha data from all_omega and all_Ep
# xnew       : linearly spaced range for spline function built on original log(omega)*alpha

log_omega = []
log_Ep = []
if MC_red == False:
	for k in list(Ep_isotherm_shifted.keys()):
		log_omega.append(np.log10(Ep_isotherm_shifted[k][0]))
		log_Ep.append(np.log10(Ep_isotherm_shifted[k][1]))
	x2, y2, xnew, ynew, poly_output = fit_poly_func(log_omega,log_Ep,num_terms)

else:
	for k in list(Ep_data_all.keys()):
		log_omega.append(np.log10(Ep_data_all[k][0]))
		log_Ep.append(np.log10(Ep_data_all[k][1]))
	x2, y2, xnew, ynew, poly_output = fit_poly_func(log_omega,log_Ep,num_terms)

# Raise frequency and storage modulus values to base 10 for Prony series fit
x3 = 10**xnew
y3 = 10**ynew
len(x2)

1769

#### Fit Prony Series to Shifted Master Curve and Generate Relaxation Modulus

$$
\text{Eq. (1) --- Storage Modulus via Relaxation Spectrum:} \hspace{10pt} E'_{discrete}(\omega) = 3\mu^{eq} + \sum^{N}_{k=1} 2.7\mu_k^{neq} \frac{\omega^2 \tau_k^2}{1+\omega^2 \tau_k^2} \hspace{10pt} \text{for k processes} \\
\text{Eq. (2) --- Storage Modulus via Prony Series Fit (Abaqus):}\hspace{10pt}\frac{G(\omega)}{G_0} = (1 - \sum^N_{i=1}g_i^p) + \sum^N_{i=1}\frac{g_i^p \tau_i^2 \omega^2}{1+\tau_i^2 \omega^2}
$$

In [380]:
# Using master curve data, find shifted storage modulus value associated with
# highest frequency which will serve as the instantaneous modulus
E0 = 10 ** y2[x2.index(max(x2))]

# Least squares fit for Prony series
Ei = fit_prony(E0,x3,y3,num_prony_terms)

# Calculate modulus ratios and write (g,k,rho) data points to file for use in Abaqus input file
#E0 = Ei[0]
G0 = E0 / (2 * (1 + nu))
K0 = E0 / (3 * (1 - 2 * nu))
Gi = Ei / (2 * (1 + nu)) 
Ki = Ei / (3 * (1 - 2 * nu))

# Calculate stress relaxation moduli
t = np.logspace(np.log10(min(p)),np.log10(max(p)),100)
E_r = relaxation_modulus_abq(p,E0,Ei,t)

# Plot stress relaxation behavior
plt.figure(10)
plt.loglog(t,E_r,'-r')
#plt.loglog(t,G_r,'-b')
#plt.loglog(t,K_r,'-g')
plt.xlabel('Time (s)')
plt.ylabel('Relaxation Modulus (MPa)')
plt.grid(which='both',axis='both')
plt.legend(['$E_{r}$'])


#### Write out data and generate Abaqus input file for material

In [381]:
# Dataframe to write out relaxation moduli values for Abaqus input file
prony_df = pd.DataFrame()
prony_df['Gi'] = Gi
prony_df['Ki'] = Ki
prony_df['tau'] = p
prony_df.to_csv(os.path.join(dir,'abaqus_prony_series.txt'),index=False)

# Write Abaqus material keyword
fid = mat + '_linear_visco_Tref' + T0 + '.inp'
filename = os.path.join(dir,fid)

prony_out = []
for i in range(0,len(Gi)-1):
    prony_out.append([round(Gi[i],4),0,p[i]])

def write_list(f, lst, n):
    for i in range(0, len(lst)):
        item = lst[i]
        f.write(",".join(str(val) for val in item) + "\n")
    
with open(filename, 'w') as f:
    line1 = str('*Material, name=' + mat)
    line2 = '*Elastic,moduli=INSTANTANEOUS'
    line3 = str(round(E0,2)) + ',' + str(nu)
    line4 = '*Viscoelastic,time=Prony'   
    f.write('{}\n{}\n{}\n{}\n'.format(line1,line2,line3,line4))
    write_list(f,prony_out,1)